# Classifying MTG Decks with Set Transformers


This notebook walks through the process of building latent embeddings for cards in Magic the Gathering, and then attempting to use the embeddings in a set transformer with multiheaded self-attention. The idea is to use a transformer to adjust card embeddings based on context and synergy within a decklist, and learn deck strengths. 

# 0. Setup

This notebook is meant to be all-encompassing. Whether it's hyperparameter search, training from scratch, or finetuning, there's some config that needs to be setup before the rest of the code is run.

In [137]:
import contextlib
import copy
import json
import random
import re
import unicodedata
import itertools
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from tqdm.auto import tqdm

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import kendalltau
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torch.utils.tensorboard import SummaryWriter


c:\Users\Kevin\Documents\GitHub\mtg-deck-evaluator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


There are 3 datasets here. The unsupervised jsonl is a huge corpus of nearly 500k magic decks spanning multiple eternal formats including modern, legacy, vintage, but mostly edh, that have been processed into sets of card names, in order to be used as "sentences" for training a word2vec model for cooccurence embeddings. The supervised jsonl is a corpus of nearly 200k moxfield edh decks, that are user/auto labelled into brackets from 1-5 on power level, 5 being as optimized as possible, 1 being absolute jank piles. These decks may have duplicate cards, despite being from a singleton format, due to cards like shadowborn apostle and persistent petitioners, as well as basic lands. This corpus is meant to train for what decks across various brackets should look like. Finally, the last corpus is around 200 cEDH tournament events where each list and its placement are kept. This is meant to test the model's predictive power be guessing the more powerful decks and checking against tournament results. However this is only improves performance for highly optimized decks as tournaments tend to attract highly optimized decks when competition is on the line.

In [ ]:
UNSUPERVISED_JSONL = Path("../../data/cooccurence/embedding_corpus.jsonl")
SUPERVISED_JSONL = Path("../../data/supervised/supervised_corpus.jsonl")
TOURNAMENT_JSONL = Path("../../data/tournament/tournament_corpus.jsonl")
GAMEKNIGHTS_JSONL = Path("../../data/gameknights_archidekt_decks.jsonl")
COMBINED_TOURNAMENT_JSONL = Path("../../data/tournament/combined_tournament_plus_gameknights.jsonl")

For preloaded embeddings, generated with embeddings model, and saving model checkpoints for later training

In [139]:
HYBRID_EMBED_PATH = Path("../embeddings/embedding-models/896dim_oracle_embeddings.pt")
CHECKPOINT_DIR = Path("./checkpoints")

These are notebook configurations, depending on how the notebook run is desired:

In [140]:
# Deck geometry
MAX_DECK_LEN = 115    # EDH decks are exactly 100 cards, but we allow 15 extra slots for companions/sideboards/wishboards.
MAX_QTY_EMBED = 50    # Caps identical card counts. Bumped to 50 to safely handle "Relentless Rats" or "Dragon's Approach" decks.
BATCH_SIZE = 64       # Number of decks processed per gradient update. 64 is the sweet spot for memory vs. gradient stability.
SEED = 42             # Locks all RNG for reproducible training runs.

# Performance & hardware optimizations
USE_AMP = True             # Automatic Mixed Precision. Drastically reduces VRAM usage and speeds up matrix multiplication.
AMP_DTYPE = "bf16"         # Bfloat16 is mathematically safer than fp16 for Transformers, preventing gradient overflow on Blackwell GPUs.
USE_ONECYCLE_LR = True     # Proactive learning rate scheduler to prevent early-stage attention collapse.
ONECYCLE_WARMUP_PCT = 0.10 # Spends the first 10% of training warming up the LR from 0 to max, then decays via cosine curve.

# Define exactly what the notebook should execute when clicking "Run All"
RUN_EMBEDDINGS = False     # Set to True to execute embeddings generation (otherwise will preload)
RUN_HPARAM_SEARCH = False  # Set to True to execute Phase A (Grid Search) on small data subsets to find optimal hyperparameters.
RUN_CURRICULUM = True      # Set to True to execute Phase B (Master Run) using the best config on the full 190k+ deck dataset.

MASTER_RUN_NAME = "set_transformer_master_run" # Filename for the final saved PyTorch weights.

DEFAULT_CONFIG = {
    # Architecture Dimensions
    "hidden_dim": 512,             # The internal reasoning space for the cards. 
    "num_heads": 8,                # Multihead attention split. 8 heads allows the model to look at 8 different card synergies simultaneously.
    "num_blocks": 4,               # Number of stacked Self-Attention layers. 4 is deep enough to learn EDH without catastrophic overfitting.
    "num_pma_seeds": 4,            # Pooling layers. 4 seeds allow the model to bucket distinct aspects (e.g., mana base, interaction, win-cons) before scoring.
    "max_qty_embed": MAX_QTY_EMBED,

    # Training Dynamics
    "batch_size": BATCH_SIZE,
    "lr_phase1": 2e-4,             # MLM Pretraining: Needs a higher LR to learn the syntax of MTG from scratch.
    "lr_phase2": 1e-4,             # Huber Regression: Moderate LR to safely anchor the 1.0 to 5.0 scale using human labels.
    "lr_phase3": 5e-5,             # Tournament Pairwise: Tiny LR to act as a magnifying glass, strictly fine-tuning the cEDH top-end.
    "dropout": 0.10,               # Regularization to prevent the attention heads from just memorizing common staple packages.
    
    # Loss Function Tuning
    "delta": 0.75,                 # Huber Loss threshold. Controls how aggressively the model ignores biased/ego-driven Moxfield labels.
    "margin": 0.10,                # Pairwise Ranking threshold. Requires the model to put at least 0.10 distance between a winning and losing tournament deck.
    "trainable_last_blocks": 1,    # During Phase 3, we freeze the foundation and only allow the final block to adjust to cEDH tournament logic.
}

# We use itertools.product to test every combination of these parameters.
grid_space = {
    "margin": [0.10, 0.05, 0.01],  # Testing whether cEDH placement gaps should be rigid (0.10) or razor-thin (0.01).
    "delta": [1.00, 0.75, 0.50],   # Testing how much we trust user labels (1.0) vs. trusting the model's own geometry (0.50).
    "dropout": [0.10, 0.2, 0.3], # Testing standard regularization vs. heavy anti-memorization.
    "lr_phase3": [5e-5, 1e-5],     # Testing normal fine-tuning vs. highly conservative micro-adjustments.
}

# Grid Search Data Budgets (Kept small34 so all 54 combinations finish reasonably)
SEARCH_SUP_MAX_ITEMS = 45000       # Only use 15k Moxfield decks per search trial.
SEARCH_TOUR_MAX_ITEMS = 100        # Only use 100 events to quickly evaluate Kendall's Tau.
SEARCH_PHASE1_EPOCHS = 2
SEARCH_PHASE2_EPOCHS = 3
SEARCH_PHASE3_EPOCHS = 1

# Used when RUN_CURRICULUM = True. This applies the winning grid search config to the entire database to forge the production-ready model.
TRAIN_SUP_MAX_ITEMS = None         # None = use all 130,000+ decks in the supervised corpus.
TRAIN_TOUR_MAX_ITEMS = None        # None = use all valid tournament events.
TRAIN_PHASE1_EPOCHS = 8            # Deep run to fully map objective MTG deck geometry.
TRAIN_PHASE2_EPOCHS = 12           # Anchors the continuous 1-5 scale. (Watch validation loss for overfitting here).
TRAIN_PHASE3_EPOCHS = 5            # Gentle calibration to natively align the top-end with cEDH tournament realities.

In [141]:
# Initialize hardware device globally so all subsequent cells can use it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Global Device set to: {device}")

Global Device set to: cuda


In [142]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. Embeddings

Embeddings are calculated by using both word2vec to use card cooccurence to guess functionality based on context, but also with a finetuned lightweight BERT for embedding rules text, placing direct information from card text into the card embedding as well.

In [143]:
if not RUN_EMBEDDINGS: #load embeddings
    blob = torch.load(HYBRID_EMBED_PATH, map_location="cpu")
    vocab = blob["vocab"]
    weights = blob["weights"].float()
    VOCAB_SIZE = len(vocab)
    CARD_EMBED_DIM = weights.shape[1]

In [144]:
if RUN_EMBEDDINGS: #train embeddings
    import gc
    import math
    from datasets import Dataset
    from gensim.models import Word2Vec
    from transformers import (
        AutoModelForMaskedLM, AutoTokenizer, AutoModel,
        DataCollatorForLanguageModeling, Trainer, TrainingArguments
    )

    BASE_SEMANTIC_MODEL = "microsoft/MiniLM-L12-H384-uncased"
    MTGJSON_FILE = Path("../../data/oracle_cards.json")
    SEMANTIC_MODEL_DIR = Path("embedding-models/mtg-minilm-mlm")
    W2V_OUTPUT = Path("../embeddings/embedding-models/w2v_mtg_cooccurrence.model")    
    
    W2V_DIM = 512
    W2V_WEIGHT = 0.4
    NLP_WEIGHT = 0.6
    
    def get_combined_oracle_text(card: Dict) -> str:
        """Combines physical stats and Oracle text into a semantic paragraph."""
        chunks = []
        name, cost, t_line = card.get("name", ""), card.get("mana_cost", ""), card.get("type_line", "")
        pt = f"{card.get('power', '')}/{card.get('toughness', '')}".strip("/")
        loyalty = card.get("loyalty", "")

        if name: chunks.append(f"Name: {name}.")
        if cost: chunks.append(f"Mana cost: {cost}.")
        if t_line: chunks.append(f"Type: {t_line}.")
        if pt: chunks.append(f"Stats: {pt}.")
        if loyalty: chunks.append(f"Loyalty: {loyalty}.")

        for face in card.get("card_faces", []):
            f_name, f_cost, f_type = face.get("name", ""), face.get("mana_cost", ""), face.get("type_line", "")
            f_pt = f"{face.get('power', '')}/{face.get('toughness', '')}".strip("/")
            f_summary = " | ".join(p for p in [f_name, f_cost, f_type] if p)
            if f_pt: f_summary += f" | Stats: {f_pt}"
            if f_summary: chunks.append(f"Face: {f_summary}.")

        text_parts = [(" ".join(chunks).strip())]
        if card.get("oracle_text"): text_parts.append(card.get("oracle_text").strip())
        for face in card.get("card_faces", []):
            if face.get("oracle_text"): text_parts.append(face.get("oracle_text").strip())

        # Deduplicate identical faces
        return "\n".join(list(dict.fromkeys(text_parts)))

    print("Parsing Oracle dictionary...")
    with open(MTGJSON_FILE, 'r', encoding='utf-8') as f:
        cards_data = json.load(f)

    oracle_index: Dict[str, str] = {}
    for card in cards_data:
        names = [card.get("name", "")] + [face.get("name", "") for face in card.get("card_faces", [])]
        for name in names:
            if not name: continue
            def normalize_card_name(name: str) -> str:
                name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii")
                name = name.lower().strip()
                name = re.sub(r"^a-", "", name)
                name = re.sub(r"\s+", " ", name)
                front_face = re.split(r"\s*//\s*", name)[0]
                return front_face.strip()
            
            norm_name = normalize_card_name(name) # Uses notebook's global function!
            combined_text = get_combined_oracle_text(card)
            
            # Save the richest text version available
            if combined_text.strip() and not oracle_index.get(norm_name, "").strip():
                oracle_index[norm_name] = combined_text


    if not SEMANTIC_MODEL_DIR.exists():
        print(f"\n--- Fine-tuning {BASE_SEMANTIC_MODEL} ---")
        texts = [text for text in oracle_index.values() if text.strip()]
        tokenizer = AutoTokenizer.from_pretrained(BASE_SEMANTIC_MODEL)
        model = AutoModelForMaskedLM.from_pretrained(BASE_SEMANTIC_MODEL)

        tokenized = Dataset.from_dict({"text": texts}).map(
            lambda b: tokenizer(b["text"], truncation=True, max_length=128),
            batched=True, remove_columns=["text"]
        )

        def group_texts(examples):
            concat = {k: sum(examples[k], []) for k in examples.keys()}
            total_len = (len(concat["input_ids"]) // 128) * 128
            return {k: [t[i: i + 128] for i in range(0, total_len, 128)] for k, t in concat.items()}

        lm_dataset = tokenized.map(group_texts, batched=True)
        data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

        training_args = TrainingArguments(
            output_dir=str(SEMANTIC_MODEL_DIR),
            num_train_epochs=2.0,
            per_device_train_batch_size=32,
            learning_rate=5e-5,
            save_strategy="no",
            report_to="none"
        )

        trainer = Trainer(model=model, args=training_args, train_dataset=lm_dataset, data_collator=data_collator)
        trainer.train()
        trainer.save_model(str(SEMANTIC_MODEL_DIR))
        tokenizer.save_pretrained(str(SEMANTIC_MODEL_DIR))
        
        # Free up VRAM
        del model, trainer, lm_dataset
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"\n✅ Semantic model already fine-tuned at {SEMANTIC_MODEL_DIR}")

    class MTGDeckCorpus:
        def __iter__(self):
            with open(UNSUPERVISED_JSONL, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip(): yield json.loads(line).get("cards", {})

    if not W2V_OUTPUT.exists():
        print("\n--- Training Gensim Word2Vec Model ---")
        W2V_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        w2v_model = Word2Vec(sentences=MTGDeckCorpus(), vector_size=W2V_DIM, window=115, min_count=3, sg=1, workers=12, epochs=10)
        w2v_model.save(str(W2V_OUTPUT))
    else:
        print(f"✅ Word2Vec model already trained at {W2V_OUTPUT}")
        w2v_model = Word2Vec.load(str(W2V_OUTPUT))

    print("\n--- Fusing Embeddings ---")
    tokenizer = AutoTokenizer.from_pretrained(str(SEMANTIC_MODEL_DIR))
    nlp_model = AutoModel.from_pretrained(str(SEMANTIC_MODEL_DIR)).to(device).eval()
    nlp_dim = nlp_model.config.hidden_size

    vocab = w2v_model.wv.index_to_key
    filtered_vocab = [c for c in vocab if oracle_index.get(normalize_card_name(c), "").strip()]
    print(f"Dropping {len(vocab) - len(filtered_vocab)} cards missing Oracle text.")

    vocab_size = len(filtered_vocab) + 2 # +2 for <PAD> and <UNK>
    hybrid_dim = W2V_DIM + nlp_dim
    hybrid_matrix = np.zeros((vocab_size, hybrid_dim))
    final_vocab_dict = {"<PAD>": 0, "<UNK>": 1}

    print("Generating NLP embeddings and concatenating...")
    # 
    for i, card_name in enumerate(filtered_vocab):
        pytorch_id = i + 2
        final_vocab_dict[card_name] = pytorch_id
        
        vec_w2v = w2v_model.wv[card_name]
        n_w2v = np.linalg.norm(vec_w2v)
        if n_w2v > 0: vec_w2v = (vec_w2v / n_w2v) * math.sqrt(W2V_WEIGHT)

        text = oracle_index.get(normalize_card_name(card_name), "")
        encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
        with torch.no_grad():
            out = nlp_model(**encoded)
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            vec_nlp = (torch.sum(out.last_hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)).squeeze(0).cpu().numpy()
            
        n_nlp = np.linalg.norm(vec_nlp)
        if n_nlp > 0: vec_nlp = (vec_nlp / n_nlp) * math.sqrt(NLP_WEIGHT)

        fused = np.concatenate([vec_w2v, vec_nlp])
        n_fused = np.linalg.norm(fused)
        if n_fused > 0: fused = fused / n_fused
            
        hybrid_matrix[pytorch_id] = fused

    # Save to disk for the Set Transformer to load
    torch.save({'weights': torch.FloatTensor(hybrid_matrix), 'vocab': final_vocab_dict}, HYBRID_EMBED_PATH)
    print(f"\n🎉 Saved Hybrid Tensor ({vocab_size}x{hybrid_dim}) to {HYBRID_EMBED_PATH}")
    
    del nlp_model, w2v_model, hybrid_matrix
    gc.collect()
    torch.cuda.empty_cache()

## 2. Data Preprocessing

We load decklists as unordered sets of card IDs with quantity IDs and a padding mask so the model can attend over variable-size sets. 



In [145]:
LOADER_NUM_WORKERS = 0
LOADER_PIN_MEMORY = True

In [146]:
def normalize_card_name(name: str) -> str:
    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode("ascii")
    name = name.lower().strip()
    name = re.sub(r"^a-", "", name)
    name = re.sub(r"\s+", " ", name)
    front_face = re.split(r"\s*//\s*", name)[0]
    return front_face.strip()

In [147]:
def extract_card_qty_role_triplets(deck_obj: Dict, card_metadata: Optional[Dict] = None) -> List[Tuple[str, int, int]]:
    triplets = []

    if "cmds" in deck_obj or "main" in deck_obj:
        for item in deck_obj.get("cmds", []):
            if isinstance(item, dict) and item.get("name"):
                triplets.append((item["name"], int(item.get("qty", 1)), 1))
        
        for zone in ("main", "mainboard", "sideboard"):
            for item in deck_obj.get(zone, []):
                if isinstance(item, dict) and item.get("name"):
                    triplets.append((item["name"], int(item.get("qty", 1)), 0))

    elif isinstance(deck_obj.get("cards"), dict):
        for name, qty in deck_obj["cards"].items():
            triplets.append((name, int(qty), 0)) 

    # Normalize and filter
    filtered = []
    for card_name, qty, role_id in triplets:
        normalized = normalize_card_name(card_name)
        if qty > 0:
            filtered.append((normalized, qty, role_id))

    return filtered

In [148]:
class MTGDeckDataset(Dataset[Dict[str, torch.Tensor]]):
    def __init__(self, jsonl_path, vocab, max_len=MAX_DECK_LEN, max_items=None):
        self.vocab = vocab
        self.max_len = max_len
        self.records = []
        
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    self.records.append(json.loads(line))
                    if max_items and len(self.records) >= max_items:
                        break

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        data = self.records[idx]
        
        label = float(data.get("bracket", 3.0))
        is_auto = bool(data.get("is_autobracket", False))

        # Extract normalized triplets: (card_name, qty, role_id)
        triplets = extract_card_qty_role_triplets(data)
        
        # Fallback: If a deck is malformed or empty, just grab the next one so the DataLoader doesn't crash on a NoneType.
        if len(triplets) < 10:
            return self.__getitem__((idx + 1) % len(self.records))

        card_ids = []
        qty_ids = []
        role_ids = []

        for card_name, qty, role in triplets:
            card_ids.append(self.vocab.get(card_name, 1)) # 1 is usually the UNK token
            qty_ids.append(max(1, min(int(qty), MAX_QTY_EMBED)))
            role_ids.append(role)

        card_ids = card_ids[: self.max_len]
        qty_ids = qty_ids[: self.max_len]
        role_ids = role_ids[: self.max_len]
        mask = [1] * len(card_ids)

        pad_len = self.max_len - len(card_ids)
        if pad_len > 0:
            card_ids += [0] * pad_len
            qty_ids += [0] * pad_len
            role_ids += [2] * pad_len  # Role 2 is the Padding Role
            mask += [0] * pad_len

        return {
            "card_ids": torch.tensor(card_ids, dtype=torch.long),
            "qty_ids": torch.tensor(qty_ids, dtype=torch.long),
            "role_ids": torch.tensor(role_ids, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.bool),
            "target": torch.tensor(label, dtype=torch.float32),
            "is_auto": torch.tensor(is_auto, dtype=torch.bool)
        }

In [ ]:
class TournamentEventDataset(Dataset[List[Dict[str, Any]]]):
    """
    Groups decks by event_url so each item is one full tournament-like event.
    Supports confidence weighting (defaults to 1.0 for legacy tournament rows).
    """

    def __init__(self, jsonl_path: Path, vocab: Dict[str, int], max_len: int = MAX_DECK_LEN):
        self.vocab = vocab
        self.max_len = max_len
        self.events = {}

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)

                event_url = data.get("event_url")
                placement = data.get("placement")
                if not event_url or placement is None:
                    continue

                try:
                    placement_value = float(placement)
                    confidence_value = float(data.get("confidence_weight", 1.0))
                except (TypeError, ValueError):
                    continue

                triplets = extract_card_qty_role_triplets(data)
                if len(triplets) < 10:
                    continue

                card_ids = []
                qty_ids = []
                role_ids = []

                for card_name, qty, role in triplets:
                    card_ids.append(self.vocab.get(card_name, 1))
                    qty_ids.append(max(1, min(int(qty), MAX_QTY_EMBED)))
                    role_ids.append(role)

                card_ids = card_ids[: self.max_len]
                qty_ids = qty_ids[: self.max_len]
                role_ids = role_ids[: self.max_len]
                mask = [1] * len(card_ids)

                pad_len = self.max_len - len(card_ids)
                if pad_len > 0:
                    card_ids += [0] * pad_len
                    qty_ids += [0] * pad_len
                    role_ids += [2] * pad_len  # Role 2 is the PAD token
                    mask += [0] * pad_len

                self.events.setdefault(str(event_url), []).append(
                    {
                        "card_ids": card_ids,
                        "qty_ids": qty_ids,
                        "role_ids": role_ids,
                        "mask": mask,
                        "placement": placement_value,
                        "confidence": confidence_value,
                    }
                )

        self.event_list = [decks for decks in self.events.values() if len(decks) >= 2]
        print(f"Loaded {len(self.event_list)} valid tournament events from {jsonl_path}")

    def __len__(self) -> int:
        return len(self.event_list)

    def __getitem__(self, idx: int):
        return self.event_list[idx]


def tournament_collate_fn(batch):
    event_decks = batch[0]

    return {
        "card_ids": torch.tensor([d["card_ids"] for d in event_decks], dtype=torch.long),
        "qty_ids": torch.tensor([d["qty_ids"] for d in event_decks], dtype=torch.long),
        "role_ids": torch.tensor([d["role_ids"] for d in event_decks], dtype=torch.long),
        "mask": torch.tensor([d["mask"] for d in event_decks], dtype=torch.bool),
        "placement": torch.tensor([d["placement"] for d in event_decks], dtype=torch.float32),
        "confidence": torch.tensor([d["confidence"] for d in event_decks], dtype=torch.float32),
    }

In [150]:
def subset_dataset(dataset: Dataset, max_items: Optional[int] = None, seed: int = SEED):
    if max_items is None or len(dataset) <= max_items:
        return dataset

    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:max_items].tolist()
    return Subset(dataset, indices)


def split_supervised_loader(
    dataset: Dataset,
    val_ratio: float = 0.1,
    batch_size: int = BATCH_SIZE,
    seed: int = SEED,
):
    if len(dataset) < 2:
        raise ValueError("Supervised dataset is too small to create train/val splits.")

    val_size = max(1, int(len(dataset) * val_ratio))
    val_size = min(val_size, len(dataset) - 1)
    train_size = len(dataset) - val_size

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=generator)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=LOADER_NUM_WORKERS,
        pin_memory=LOADER_PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=LOADER_NUM_WORKERS,
        pin_memory=LOADER_PIN_MEMORY,
    )
    return train_loader, val_loader


def split_tournament_loaders(
    dataset: Dataset,
    val_ratio: float = 0.2,
    seed: int = SEED,
):
    
    val_size = max(1, int(len(dataset) * val_ratio))
    val_size = min(val_size, len(dataset) - 1)
    train_size = len(dataset) - val_size

    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=generator)
    train_loader = DataLoader(
        train_ds,
        batch_size=1,
        shuffle=True,
        collate_fn=tournament_collate_fn,
        num_workers=LOADER_NUM_WORKERS,
        pin_memory=LOADER_PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=tournament_collate_fn,
        num_workers=LOADER_NUM_WORKERS,
        pin_memory=LOADER_PIN_MEMORY,
    )
    return train_loader, val_loader

In [ ]:
def combine_event_jsonl(input_paths: List[Path], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    seen_pairs = set()
    written = 0

    with output_path.open("w", encoding="utf-8") as out_f:
        for path in input_paths:
            if not path.exists():
                print(f"[WARN] Missing input file: {path}")
                continue

            with path.open("r", encoding="utf-8") as in_f:
                for line in in_f:
                    if not line.strip():
                        continue

                    row = json.loads(line)
                    key = (str(row.get("event_url")), str(row.get("deck_url")))
                    if key in seen_pairs:
                        continue
                    seen_pairs.add(key)
                    out_f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    written += 1

    print(f"Combined event rows written: {written} -> {output_path}")


combine_event_jsonl(
    [TOURNAMENT_JSONL, GAMEKNIGHTS_JSONL],
    COMBINED_TOURNAMENT_JSONL,
 )

print("Loading Supervised Corpus...")
sup_dataset = MTGDeckDataset(
    jsonl_path=SUPERVISED_JSONL,
    vocab=vocab,
    max_len=MAX_DECK_LEN
 )

print("Loading Tournament + GameKnights Corpus...")
tour_dataset = TournamentEventDataset(
    jsonl_path=COMBINED_TOURNAMENT_JSONL,
    vocab=vocab,
    max_len=MAX_DECK_LEN
 )

print("\nCreating DataLoaders...")
sup_train_loader, sup_val_loader = split_supervised_loader(
    sup_dataset,
    val_ratio=0.1,
    batch_size=BATCH_SIZE
 )

tour_train_loader, tour_eval_loader = split_tournament_loaders(
    tour_dataset,
    val_ratio=0.2
 )

print("-" * 40)
print(f"Supervised Train Batches: {len(sup_train_loader)} | Val Batches: {len(sup_val_loader)}")
print(f"Tournament+GK Train Events: {len(tour_train_loader)} | Eval Events: {len(tour_eval_loader)}")
print("-" * 40)

Loading Supervised Corpus...
Loading Tournament Corpus...
Loaded 935 valid tournament events from ..\..\data\tournament\tournament_corpus.jsonl

Creating DataLoaders...
----------------------------------------
Supervised Train Batches: 3011 | Val Batches: 335
Tournament Train Events:  748 | Eval Events: 187
----------------------------------------


## 3. Model Architecture - Set Transformer

The architecture here uses multiheaded attention layers

In [152]:
class SetTransformer(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_heads: int, num_blocks: int, num_pma_seeds: int, max_qty_embed: int, dropout: float = 0.1, pretrained_weights: Optional[torch.Tensor] = None):
        super().__init__()
        
        # 1. Embeddings
        self.card_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_weights is not None:
            self.card_embedding.weight.data.copy_(pretrained_weights)
            
        self.qty_embedding = nn.Embedding(max_qty_embed + 1, embed_dim, padding_idx=0) 
        self.role_embedding = nn.Embedding(3, embed_dim, padding_idx=2) # 0=Main, 1=Cmd, 2=Pad

        # 2. Transformer Encoder (Note: batch_first=True removes the need to transpose!)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim * 3, 
            nhead=num_heads, 
            dim_feedforward=hidden_dim, 
            dropout=dropout,
            batch_first=True 
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        
        # 3. Pooling by Multihead Attention (PMA)
        self.pma_seeds = nn.Parameter(torch.randn(num_pma_seeds, embed_dim * 3))
        self.pma_attention = nn.MultiheadAttention(
            embed_dim=embed_dim * 3, 
            num_heads=num_heads, 
            batch_first=True, 
            dropout=dropout
        )
        
        # 4. Output Heads
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(embed_dim * 3 * num_pma_seeds, 1) # Phase 2 & 3
        self.mlm_head = nn.Linear(embed_dim * 3, vocab_size)            # Phase 1
        
    def encode_tokens(self, card_ids: torch.Tensor, qty_ids: torch.Tensor, role_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        card_embeds = self.card_embedding(card_ids)
        qty_embeds = self.qty_embedding(qty_ids)
        role_embeds = self.role_embedding(role_ids)

        x = torch.cat([card_embeds, qty_embeds, role_embeds], dim=-1)
        transformer_out = self.transformer_encoder(x, src_key_padding_mask=~mask)
        return transformer_out

    def forward_mlm(self, card_ids: torch.Tensor, qty_ids: torch.Tensor, role_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # Used ONLY in Phase 1
        transformer_out = self.encode_tokens(card_ids, qty_ids, role_ids, mask)
        return self.mlm_head(transformer_out)

    def forward(self, card_ids: torch.Tensor, qty_ids: torch.Tensor, role_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # Used in Phase 2, Phase 3, and Inference
        transformer_out = self.encode_tokens(card_ids, qty_ids, role_ids, mask)
        batch_size = card_ids.size(0)
        
        pma_seeds_expanded = self.pma_seeds.unsqueeze(0).expand(batch_size, -1, -1)

        pma_out, _ = self.pma_attention(
            query=pma_seeds_expanded, 
            key=transformer_out, 
            value=transformer_out, 
            key_padding_mask=~mask 
        )

        pma_out_flat = pma_out.reshape(batch_size, -1) 
        pma_out_flat = self.dropout(pma_out_flat)

        # Force the output into a 1.0 to 5.0 scale for the brackets
        raw = self.output_layer(pma_out_flat).squeeze(-1)
        return 1.0 + 4.0 * torch.sigmoid(raw)

In [153]:
model = SetTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=CARD_EMBED_DIM,
    hidden_dim=DEFAULT_CONFIG["hidden_dim"],
    num_heads=DEFAULT_CONFIG["num_heads"],
    num_blocks=DEFAULT_CONFIG["num_blocks"],
    num_pma_seeds=DEFAULT_CONFIG["num_pma_seeds"],
    max_qty_embed=DEFAULT_CONFIG["max_qty_embed"],
    dropout=DEFAULT_CONFIG["dropout"]
).to(device)

## 4. Training

Here we run the 3 phases of training, but first, some evaluation metrics and build:

In [154]:
def build_model(config: Dict[str, Any]) -> SetTransformer:
    return SetTransformer(
        vocab_size=VOCAB_SIZE,
        embed_dim=CARD_EMBED_DIM,
        hidden_dim=int(config.get("hidden_dim", 512)),
        num_heads=int(config.get("num_heads", 8)),
        num_blocks=int(config.get("num_blocks", 4)),
        num_pma_seeds=int(config.get("num_pma_seeds", 4)),
        max_qty_embed=int(config.get("max_qty_embed", MAX_QTY_EMBED)),
        dropout=float(config.get("dropout", 0.1)),
        pretrained_weights=weights # Passed in from Section 1
    ).to(device)

def compute_tournament_tau_stats(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    """
    Evaluates the model against the cEDH tournament dataset using Kendall's Rank Correlation.
    """
    model.eval()
    all_taus = []

    with torch.no_grad():
        for batch in loader:
            card_ids = batch["card_ids"].to(device)
            qty_ids = batch["qty_ids"].to(device)
            role_ids = batch["role_ids"].to(device)
            mask = batch["mask"].to(device)
            actual_placements = batch["placement"].cpu().numpy()

            # Kendall Tau requires at least 3 items to be mathematically meaningful
            if len(actual_placements) < 3:
                continue

            # Predict deck power
            predicted_scores = model(card_ids, qty_ids, role_ids, mask).squeeze().cpu().numpy()
            
            # Negate actual placements because lower placement (1st) is better (higher predicted score)
            tau, _ = kendalltau(predicted_scores, -actual_placements)
            if not np.isnan(tau):
                all_taus.append(float(tau))

    if not all_taus:
        return {"mean_tau": float("nan"), "median_tau": float("nan"), "positive_corr_pct": 0.0}

    tau_array = np.array(all_taus, dtype=np.float32)
    return {
        "mean_tau": float(tau_array.mean()),
        "median_tau": float(np.median(tau_array)),
        "positive_corr_pct": float((tau_array > 0).mean() * 100.0),
    }

# Quick sanity check initialization
test_model = build_model(DEFAULT_CONFIG)
print(f"Model initialized with {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,} trainable parameters.")

Model initialized with 263,805,645 trainable parameters.


### Phase 1: Decklist Pre-training (Unsupervised MLM)

Before the model tries to predict a single power level score, it needs to learn how MTG cards relate to each other. 
In this phase, we take an EDH deck, randomly mask out 15% of the cards, and force the model to guess what is missing. 

**Why we do this:**
By guessing missing cards, the Self-Attention heads learn competitive deckbuilding patterns and combo packages (e.g., if Thassa's Oracle and Demonic Consultation are in the deck, the masked card is likely Force of Will, not Colossal Dreadmaw). This builds a powerful "synergy engine" before we ever introduce the subjective 1-5 power brackets.

In [155]:
def apply_mlm_mask(card_ids: torch.Tensor, mask: torch.Tensor, mask_token_id: int = 1, mask_ratio: float = 0.15):
    # Mask token ID 1 is the <UNK> token in our vocab, acting as our generic mask
    masked_ids = card_ids.clone()
    labels = torch.full_like(card_ids, fill_value=-100) # -100 is ignored by CrossEntropyLoss

    for i in range(card_ids.size(0)):
        valid_positions = torch.nonzero(mask[i], as_tuple=False).flatten()
        if valid_positions.numel() == 0:
            continue

        n_mask = max(1, int(valid_positions.numel() * mask_ratio))
        perm = torch.randperm(valid_positions.numel(), device=card_ids.device)[:n_mask]
        chosen = valid_positions[perm]

        labels[i, chosen] = card_ids[i, chosen]
        masked_ids[i, chosen] = mask_token_id

    return masked_ids, labels

def train_phase1_mlm(model: nn.Module, loader: DataLoader, epochs: int, lr: float):
    print("\n--- Starting Phase 1: Decklist MLM Pre-training ---")
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        
        # Initialize the progress bar for the epoch
        progress_bar = tqdm(loader, desc=f"[Phase 1 MLM] Epoch {epoch + 1}/{epochs}")
        
        for batch in progress_bar:
            card_ids = batch["card_ids"].to(device)
            qty_ids = batch["qty_ids"].to(device)
            role_ids = batch["role_ids"].to(device)
            mask = batch["mask"].to(device)

            masked_ids, labels = apply_mlm_mask(card_ids, mask)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16 if AMP_DTYPE == "bf16" else torch.float16, enabled=USE_AMP):
                # Pass through the MLM head
                logits = model.forward_mlm(masked_ids, qty_ids, role_ids, mask)
                loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += float(loss.item())
            
            # Update the progress bar text with the live loss
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        print(f"Phase 1 | Epoch {epoch + 1} Completed | Avg Loss: {running_loss / len(loader):.4f}")

### Phase 2: Power Bracket Anchoring (Supervised Huber Regression)

Now that the model understands MTG synergy, we swap the MLM head for the regression output layer. 
We train the model on the Moxfield dataset to predict the 1.0 to 5.0 power bracket labels.

**Why Huber Loss?**
Crowdsourced power levels are noisy and highly subjective (everyone thinks their deck is a "7/10"). Huber Loss acts like Mean Squared Error (MSE) for small errors, but switches to Mean Absolute Error (MAE) for large errors. This prevents the model from destroying its learned synergy engine just to satisfy a single user who incorrectly labeled their cEDH deck as a 1.0 jank pile.

In [156]:
def evaluate_regression(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> float:
    model.eval()
    total = 0.0
    with torch.no_grad():
        for batch in loader:
            card_ids = batch["card_ids"].to(device)
            qty_ids = batch["qty_ids"].to(device)
            role_ids = batch["role_ids"].to(device)
            mask = batch["mask"].to(device)
            target = batch["target"].to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16 if AMP_DTYPE == "bf16" else torch.float16, enabled=USE_AMP):
                pred = model(card_ids, qty_ids, role_ids, mask)
                loss = loss_fn(pred, target)
            total += float(loss.item())
    model.train()
    return total / max(1, len(loader))

def train_phase2_huber(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int, lr: float, delta: float):
    print("\n--- Starting Phase 2: Supervised Huber Regression ---")
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    train_loss_fn = nn.HuberLoss(delta=delta, reduction="none")
    eval_loss_fn = nn.HuberLoss(delta=delta, reduction="mean")
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model.train()

    for epoch in range(epochs):
        train_loss = 0.0
        
        # Initialize the progress bar for the epoch
        progress_bar = tqdm(train_loader, desc=f"[Phase 2 Huber] Epoch {epoch + 1}/{epochs}")
        
        for batch in progress_bar:
            card_ids = batch["card_ids"].to(device)
            qty_ids = batch["qty_ids"].to(device)
            role_ids = batch["role_ids"].to(device)
            mask = batch["mask"].to(device)
            target = batch["target"].to(device)
            
            # Penalize auto-bracketed labels slightly to trust manual labels more
            is_auto = batch["is_auto"].to(device)
            weights = torch.where(is_auto, 0.5, 1.0)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16 if AMP_DTYPE == "bf16" else torch.float16, enabled=USE_AMP):
                pred = model(card_ids, qty_ids, role_ids, mask)
                unreduced_loss = train_loss_fn(pred, target)
                loss = (unreduced_loss * weights).mean()

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += float(loss.item())
            
            # Update the progress bar text with the live loss
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        val_loss = evaluate_regression(model, val_loader, eval_loss_fn)
        print(f"Phase 2 | Epoch {epoch + 1} | Train Loss: {train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

### Phase 3: cEDH Tournament Calibration (Pairwise Ranking)

Human labels can only get us so far. To truly understand the peak of competitive MTG, we freeze the base representation layers and only train the final output block. 

We feed the model decks from actual cEDH tournaments. Instead of predicting a specific number, we use a Pairwise Margin Loss: if Deck A placed higher than Deck B in the tournament, the model's predicted score for Deck A *must* be higher than Deck B by a specific margin. 

**Why we do this:**
This natively calibrates the very top end of our 1-5 scale to reflect actual tournament winning percentages, creating objective separation between "High Power Casual" and true "cEDH".

In [ ]:
def train_phase3_tournament(model: nn.Module, train_loader: DataLoader, epochs: int, lr: float, margin: float):
    print("\n--- Starting Phase 3: Tournament Pairwise Calibration ---")

    # Freeze embeddings so Phase 3 calibrates ranking behavior without erasing Phase 1/2 structure.
    for param in model.card_embedding.parameters():
        param.requires_grad = False

    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    model.train()

    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_events = 0

        progress_bar = tqdm(train_loader, desc=f"[Phase 3 Pairwise] Epoch {epoch + 1}/{epochs}")

        for batch in progress_bar:
            card_ids = batch["card_ids"].to(device)
            qty_ids = batch["qty_ids"].to(device)
            role_ids = batch["role_ids"].to(device)
            mask = batch["mask"].to(device)
            placements = batch["placement"].to(device)
            confidences = batch["confidence"].to(device)

            num_decks = card_ids.size(0)
            if num_decks < 2:
                continue

            with torch.amp.autocast("cuda", dtype=torch.bfloat16 if AMP_DTYPE == "bf16" else torch.float16, enabled=USE_AMP):
                scores = model(card_ids, qty_ids, role_ids, mask)

                pod_loss = torch.tensor(0.0, device=device)
                pairs_in_pod = 0

                # Compare each unordered pair once to avoid double-counting.
                for i in range(num_decks):
                    for j in range(i + 1, num_decks):
                        if placements[i] == placements[j]:
                            continue

                        if placements[i] < placements[j]:
                            win_idx, lose_idx = i, j
                        else:
                            win_idx, lose_idx = j, i

                        pair_loss = torch.relu(margin - (scores[win_idx] - scores[lose_idx]))
                        pair_confidence = (confidences[win_idx] + confidences[lose_idx]) / 2.0

                        pod_loss += pair_loss * pair_confidence
                        pairs_in_pod += 1

                if pairs_in_pod == 0:
                    continue

                pod_loss = pod_loss / pairs_in_pod

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(pod_loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += float(pod_loss.item())
            epoch_events += 1

            progress_bar.set_postfix({"loss": f"{epoch_loss / max(1, epoch_events):.4f}"})

        stats = compute_tournament_tau_stats(model, tour_eval_loader)
        print(
            f"Phase 3 | Epoch {epoch + 1} | Loss: {epoch_loss / max(1, epoch_events):.4f} "
            f"| Kendall Tau: {stats['mean_tau']:.3f} | Pos Corr: {stats['positive_corr_pct']:.1f}%"
        )

In [ ]:
# Execute Curriculum if enabled in Section 0
if RUN_CURRICULUM:
    # 1. Initialize fresh model
    master_model = build_model(DEFAULT_CONFIG)
    
    # 2. Phase 1: Unsupervised MLM
    # We use the supervised dataset's loader, but ignore the labels for MLM
    train_phase1_mlm(
        master_model, 
        sup_train_loader, 
        epochs=TRAIN_PHASE1_EPOCHS, 
        lr=DEFAULT_CONFIG["lr_phase1"]
    )
    
    # 3. Phase 2: Supervised Regression
    train_phase2_huber(
        master_model, 
        sup_train_loader, 
        sup_val_loader, 
        epochs=TRAIN_PHASE2_EPOCHS, 
        lr=DEFAULT_CONFIG["lr_phase2"],
        delta=DEFAULT_CONFIG["delta"]
    )
    
    # 4. Phase 3: Tournament Calibration
    train_phase3_tournament(
        master_model, 
        tour_train_loader, 
        epochs=TRAIN_PHASE3_EPOCHS, 
        lr=DEFAULT_CONFIG["lr_phase3"],
        margin=DEFAULT_CONFIG["margin"]
    )
    
    # 5. Save final weights
    final_path = CHECKPOINT_DIR / f"{MASTER_RUN_NAME}.pt"
    torch.save({
        "model_state_dict": master_model.state_dict(),
        "config": DEFAULT_CONFIG,
        "vocab_size": VOCAB_SIZE,
        "max_qty": MAX_QTY_EMBED
    }, final_path)
    
    print(f"\n🎉 Master Curriculum Complete! Model saved to {final_path}")


--- Starting Phase 1: Decklist MLM Pre-training ---


[Phase 1 MLM] Epoch 1/8:  49%|████▉     | 1479/3011 [1:12:14<2:13:30,  5.23s/it, loss=7.8603]